In [1]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [optuna]2m3/4 [optuna]


In [1]:
# ============================================================================
# Phase 1: 데이터 준비
# ============================================================================

import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import GroupKFold
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_recall_curve,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score
)
import optuna
from optuna.pruners import MedianPruner
import warnings
from datetime import datetime
import json
import pickle
import time

warnings.filterwarnings('ignore')

print("="*80)
print("🚀 부도 예측 모델: Time-based + Optuna 최적화")
print("="*80)
print(f"시작 시간: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

# 타임스탬프
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
start_time = time.time()

# ============================================================================
# Step 1: 데이터 로드
# ============================================================================

print("\n" + "="*80)
print("Step 1: 데이터 로드 및 전처리")
print("="*80)

df = pd.read_csv('./data/최종_통합데이터_모델링용.csv', encoding='utf-8-sig')
print(f"✅ 원본 데이터: {len(df):,}개 샘플")
print(f"   - 컬럼 수: {len(df.columns)}개")
print(f"   - 고유 기업: {df['Company_Name'].nunique():,}개")
print(f"   - 기간: {df['bsns_year'].min()}~{df['bsns_year'].max()}")

# Target 분포
print(f"\nTarget 분포:")
print(f"   - 정상 (0): {(df['Target']==0).sum():,}개 ({(df['Target']==0).sum()/len(df)*100:.1f}%)")
print(f"   - 부도 (1): {(df['Target']==1).sum():,}개 ({(df['Target']==1).sum()/len(df)*100:.1f}%)")

# ============================================================================
# Step 2: Time-based Split (2015~2023 Train / 2024~2025 Test)
# ============================================================================

print("\n" + "="*80)
print("Step 2: Time-based Train/Test Split")
print("="*80)

# NaN corp_code 제거
df_model = df[~df['corp_code'].isna()].copy()
print(f"corp_code 정제: {len(df_model):,}개 샘플")

# Time-based Split
df_train = df_model[df_model['bsns_year'] <= 2023].copy()
df_test = df_model[df_model['bsns_year'] >= 2024].copy()

print(f"\n✅ Train (2015~2023): {len(df_train):,}개")
print(f"   - 정상: {(df_train['Target']==0).sum():,}개 ({(df_train['Target']==0).sum()/len(df_train)*100:.1f}%)")
print(f"   - 부도: {(df_train['Target']==1).sum():,}개 ({(df_train['Target']==1).sum()/len(df_train)*100:.1f}%)")
print(f"   - 고유 기업: {df_train['corp_code'].nunique():,}개")

print(f"\n✅ Test (2024~2025): {len(df_test):,}개")
print(f"   - 정상: {(df_test['Target']==0).sum():,}개 ({(df_test['Target']==0).sum()/len(df_test)*100:.1f}%)")
print(f"   - 부도: {(df_test['Target']==1).sum():,}개 ({(df_test['Target']==1).sum()/len(df_test)*100:.1f}%)")
print(f"   - 고유 기업: {df_test['corp_code'].nunique():,}개")

# ============================================================================
# Step 3: 피처 준비
# ============================================================================

print("\n" + "="*80)
print("Step 3: 피처 준비")
print("="*80)

# 제외 컬럼
exclude_cols = [
    'stock_code', 'corp_code', 'bsns_year', 'target_year',
    'Company_Name', 'company_name', 'Status', 'Target', 'y'
]

# 피처 추출
feature_cols = [col for col in df_model.columns if col not in exclude_cols]
print(f"피처 수: {len(feature_cols)}개")

# X, y, groups
X_train = df_train[feature_cols].copy()
y_train = df_train['Target'].copy()
groups_train = df_train['corp_code'].astype(int).values

X_test = df_test[feature_cols].copy()
y_test = df_test['Target'].copy()

# inf 값 처리
print(f"\ninf 값 처리:")
inf_count_train = np.isinf(X_train).sum().sum()
inf_count_test = np.isinf(X_test).sum().sum()
X_train = X_train.replace([np.inf, -np.inf], np.nan)
X_test = X_test.replace([np.inf, -np.inf], np.nan)
print(f"   - Train: {inf_count_train:,}개 → 0개")
print(f"   - Test: {inf_count_test:,}개 → 0개")

# scale_pos_weight 계산
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f"\nscale_pos_weight: {scale_pos_weight:.2f}")

print(f"\n✅ 데이터 준비 완료!")
print(f"   - X_train: {X_train.shape}")
print(f"   - X_test: {X_test.shape}")
print(f"   - 고유 기업 (Train): {len(np.unique(groups_train)):,}개")

# ============================================================================
# Phase 2: Optuna 최적화
# ============================================================================

print("\n" + "="*80)
print("Phase 2: Optuna 하이퍼파라미터 최적화")
print("="*80)
print(f"⏱️  제한 시간: 2시간 (7200초)")
print(f"🔍 평가 지표: PR AUC (Primary) + F1 (Secondary)")
print(f"✂️  Pruning: MedianPruner")
print(f"⚡ 병렬화: n_jobs=-1\n")

# Objective 함수 정의
def objective(trial):
    """Optuna Objective: GroupKFold 3-fold CV로 PR AUC 최대화"""
    
    # 하이퍼파라미터 제안
    params = {
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'gamma': trial.suggest_float('gamma', 1e-8, 1.0, log=True),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 5.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 5.0, log=True),
        'n_estimators': trial.suggest_int('n_estimators', 100, 500),
        
        # 고정 파라미터
        'objective': 'binary:logistic',
        'eval_metric': 'aucpr',
        'tree_method': 'hist',
        'random_state': 42,
        'early_stopping_rounds': 30,
        'scale_pos_weight': scale_pos_weight
    }
    
    # GroupKFold 3-fold CV
    gkf = GroupKFold(n_splits=3)
    pr_aucs = []
    f1_scores = []
    
    for fold, (train_idx, val_idx) in enumerate(gkf.split(X_train, y_train, groups_train)):
        X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
        
        # 모델 학습
        model = xgb.XGBClassifier(**params)
        model.fit(
            X_tr, y_tr,
            eval_set=[(X_val, y_val)],
            verbose=False
        )
        
        # 예측
        y_pred_proba = model.predict_proba(X_val)[:, 1]
        
        # PR AUC
        pr_auc = average_precision_score(y_val, y_pred_proba)
        pr_aucs.append(pr_auc)
        
        # F1 (최적 threshold)
        precision, recall, thresholds = precision_recall_curve(y_val, y_pred_proba)
        f1 = 2 * (precision * recall) / (precision + recall + 1e-10)
        optimal_idx = np.argmax(f1)
        optimal_threshold = thresholds[optimal_idx] if optimal_idx < len(thresholds) else 0.5
        y_pred = (y_pred_proba >= optimal_threshold).astype(int)
        f1_scores.append(f1_score(y_val, y_pred, zero_division=0))
        
        # Pruning (fold별 중간 보고)
        trial.report(np.mean(pr_aucs), fold)
        if trial.should_prune():
            raise optuna.TrialPruned()
    
    # CV 평균 PR AUC 반환
    return np.mean(pr_aucs)

# Study 생성
study = optuna.create_study(
    direction='maximize',
    pruner=MedianPruner(n_startup_trials=10, n_warmup_steps=1),
    study_name='bankruptcy_prediction_optuna'
)

# 현재 최적 파라미터를 첫 trial로 추가 (warm start)
initial_params = {
    'learning_rate': 0.15,
    'max_depth': 9,
    'min_child_weight': 7,      # ✅ 수정
    'subsample': 1.0,
    'colsample_bytree': 0.8,    # ✅ 수정
    'gamma': 0.1,
    'reg_alpha': 0.0,           # ✅ 수정 (0.0은 1e-8로 자동 변환)
    'reg_lambda': 1.0,          # ✅ 수정
    'n_estimators': 200
}
# reg_alpha가 0.0이면 Optuna의 log scale에서 문제가 될 수 있으므로
# 매우 작은 값으로 변경
if initial_params['reg_alpha'] == 0.0:
    initial_params['reg_alpha'] = 1e-8
if initial_params['reg_lambda'] == 0.0:
    initial_params['reg_lambda'] = 1e-8

study.enqueue_trial(initial_params)
# 최적화 실행 (2시간 제한)
print(f"🚀 최적화 시작... (시작 시각: {datetime.now().strftime('%H:%M:%S')})")
study.optimize(
    objective,
    n_trials=1000,  # 최대 1000회 (시간 제한이 먼저 걸림)
    timeout=7200,   # 2시간
    n_jobs=-1,      # 모든 CPU 사용
    show_progress_bar=True
)

optuna_time = time.time() - start_time
print(f"\n✅ Optuna 최적화 완료!")
print(f"   - 소요 시간: {optuna_time/60:.1f}분")
print(f"   - 평가한 Trial 수: {len(study.trials)}개")
print(f"   - 완료된 Trial 수: {len([t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE])}개")
print(f"   - Pruned Trial 수: {len([t for t in study.trials if t.state == optuna.trial.TrialState.PRUNED])}개")

# 최적 파라미터
best_params = study.best_params.copy()
best_params.update({
    'objective': 'binary:logistic',
    'eval_metric': 'aucpr',
    'tree_method': 'hist',
    'random_state': 42,
    'early_stopping_rounds': 30,
    'scale_pos_weight': scale_pos_weight
})

print(f"\n" + "="*80)
print("🏆 최적 하이퍼파라미터")
print("="*80)
for k, v in study.best_params.items():
    print(f"   {k:20s}: {v}")

print(f"\nCV 성능 (2015~2023):")
print(f"   PR AUC: {study.best_value:.4f}")

# ============================================================================
# Phase 3: 최종 평가
# ============================================================================

print("\n" + "="*80)
print("Phase 3: 최종 모델 학습 및 평가")
print("="*80)

# 전체 Train 데이터로 학습
final_model = xgb.XGBClassifier(**best_params)
print(f"학습 중... (전체 Train 데이터)")
final_model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=False
)
print(f"✅ 학습 완료!")

# Test 예측
y_pred_proba = final_model.predict_proba(X_test)[:, 1]

# 최적 threshold (F1 최대화)
precision, recall, thresholds = precision_recall_curve(y_test, y_pred_proba)
f1_scores_threshold = 2 * (precision * recall) / (precision + recall + 1e-10)
optimal_idx = np.argmax(f1_scores_threshold)
optimal_threshold = thresholds[optimal_idx] if optimal_idx < len(thresholds) else 0.5
y_pred = (y_pred_proba >= optimal_threshold).astype(int)

# 지표 계산
test_pr_auc = average_precision_score(y_test, y_pred_proba)
test_roc_auc = roc_auc_score(y_test, y_pred_proba)
test_f1 = f1_score(y_test, y_pred)
test_precision = precision_score(y_test, y_pred)
test_recall = recall_score(y_test, y_pred)

# ============================================================================
# 최종 리포트
# ============================================================================

print("\n" + "="*80)
print("📊 최종 리포트")
print("="*80)

report = f"""
=== Optuna 최적화 완료 ===
소요 시간: {optuna_time/60:.1f}분
평가한 Trial 수: {len(study.trials)}개
완료된 Trial 수: {len([t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE])}개

=== 최적 하이퍼파라미터 ===
"""
for k, v in study.best_params.items():
    report += f"{k}: {v}\n"

report += f"""
=== CV 성능 (2015~2023) ===
PR AUC: {study.best_value:.4f}

=== Test 성능 (2024~2025) ===
PR AUC: {test_pr_auc:.4f}
ROC AUC: {test_roc_auc:.4f}
F1: {test_f1:.4f}
Precision: {test_precision:.4f}
Recall: {test_recall:.4f}
Threshold: {optimal_threshold:.4f}

=== 혼동 행렬 ===
"""

# 혼동 행렬
cm = confusion_matrix(y_test, y_pred)
report += f"""            예측 정상  예측 부도
실제 정상      {cm[0,0]:>6d}    {cm[0,1]:>6d}
실제 부도      {cm[1,0]:>6d}    {cm[1,1]:>6d}
"""

print(report)

# 분류 보고서
print("\n분류 보고서:")
print(classification_report(y_test, y_pred, target_names=['정상', '부도']))

# ============================================================================
# 파일 저장
# ============================================================================

print("\n" + "="*80)
print("💾 파일 저장")
print("="*80)

# 1. 최적 파라미터 JSON
params_file = f'best_params_optuna_{timestamp}.json'
with open(params_file, 'w', encoding='utf-8') as f:
    json.dump({
        'best_params': study.best_params,
        'cv_pr_auc': float(study.best_value),
        'test_performance': {
            'pr_auc': float(test_pr_auc),
            'roc_auc': float(test_roc_auc),
            'f1': float(test_f1),
            'precision': float(test_precision),
            'recall': float(test_recall),
            'threshold': float(optimal_threshold)
        },
        'optuna_info': {
            'n_trials': len(study.trials),
            'n_complete': len([t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]),
            'n_pruned': len([t for t in study.trials if t.state == optuna.trial.TrialState.PRUNED]),
            'optimization_time_minutes': optuna_time/60
        }
    }, f, indent=2, ensure_ascii=False)
print(f"✅ {params_file}")

# 2. Optuna Study 객체
study_file = f'optuna_study_{timestamp}.pkl'
with open(study_file, 'wb') as f:
    pickle.dump(study, f)
print(f"✅ {study_file}")

# 3. 최종 모델
model_file = f'final_model_optuna_{timestamp}.json'
final_model.save_model(model_file)
print(f"✅ {model_file}")

# 4. 리포트 텍스트
report_file = f'optimization_report_{timestamp}.txt'
with open(report_file, 'w', encoding='utf-8') as f:
    f.write(report)
print(f"✅ {report_file}")

# 5. 최적화 히스토리 시각화
try:
    import matplotlib.pyplot as plt
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # 왼쪽: Trial별 PR AUC
    trial_numbers = [t.number for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
    trial_values = [t.value for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
    axes[0].plot(trial_numbers, trial_values, 'o-', alpha=0.6)
    axes[0].axhline(y=study.best_value, color='r', linestyle='--', label=f'Best: {study.best_value:.4f}')
    axes[0].set_xlabel('Trial Number')
    axes[0].set_ylabel('PR AUC (CV)')
    axes[0].set_title('Optimization History')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # 오른쪽: 파라미터 중요도
    try:
        importances = optuna.importance.get_param_importances(study)
        params_sorted = sorted(importances.items(), key=lambda x: x[1], reverse=True)
        param_names = [p[0] for p in params_sorted[:8]]
        param_values = [p[1] for p in params_sorted[:8]]
        
        axes[1].barh(param_names, param_values)
        axes[1].set_xlabel('Importance')
        axes[1].set_title('Parameter Importance')
        axes[1].grid(True, alpha=0.3, axis='x')
    except:
        axes[1].text(0.5, 0.5, 'Not enough trials\nfor importance', 
                    ha='center', va='center', transform=axes[1].transAxes)
    
    plt.tight_layout()
    viz_file = f'optimization_history_{timestamp}.png'
    plt.savefig(viz_file, dpi=150, bbox_inches='tight')
    print(f"✅ {viz_file}")
    plt.close()
except Exception as e:
    print(f"⚠️  시각화 생성 실패: {e}")

# ============================================================================
# 완료
# ============================================================================

total_time = time.time() - start_time
print("\n" + "="*80)
print("🎉 전체 작업 완료!")
print("="*80)
print(f"총 소요 시간: {total_time/60:.1f}분")
print(f"종료 시간: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

# 성공 기준 체크
print("\n" + "="*80)
print("✅ 성공 기준 체크")
print("="*80)
print(f"{'✅' if total_time <= 7200 else '❌'} 2시간 이내 완료: {total_time/60:.1f}분")
print(f"{'✅' if test_pr_auc >= 0.65 else '❌'} Test PR AUC > 0.65: {test_pr_auc:.4f}")
print(f"✅ 모든 출력 파일 생성 완료")
print(f"✅ 에러 없이 실행 완료")

print("\n" + "="*80)


[I 2025-12-24 19:18:43,904] A new study created in memory with name: bankruptcy_prediction_optuna


🚀 부도 예측 모델: Time-based + Optuna 최적화
시작 시간: 2025-12-24 19:18:43


Step 1: 데이터 로드 및 전처리
✅ 원본 데이터: 19,603개 샘플
   - 컬럼 수: 57개
   - 고유 기업: 2,801개
   - 기간: 2015~2025

Target 분포:
   - 정상 (0): 15,616개 (79.7%)
   - 부도 (1): 3,987개 (20.3%)

Step 2: Time-based Train/Test Split
corp_code 정제: 19,555개 샘플

✅ Train (2015~2023): 17,037개
   - 정상: 13,443개 (78.9%)
   - 부도: 3,594개 (21.1%)
   - 고유 기업: 2,609개

✅ Test (2024~2025): 2,518개
   - 정상: 2,127개 (84.5%)
   - 부도: 391개 (15.5%)
   - 고유 기업: 2,472개

Step 3: 피처 준비
피처 수: 49개

inf 값 처리:
   - Train: 149개 → 0개
   - Test: 40개 → 0개

scale_pos_weight: 3.74

✅ 데이터 준비 완료!
   - X_train: (17037, 49)
   - X_test: (2518, 49)
   - 고유 기업 (Train): 2,609개

Phase 2: Optuna 하이퍼파라미터 최적화
⏱️  제한 시간: 2시간 (7200초)
🔍 평가 지표: PR AUC (Primary) + F1 (Secondary)
✂️  Pruning: MedianPruner
⚡ 병렬화: n_jobs=-1

🚀 최적화 시작... (시작 시각: 19:18:43)


  0%|          | 0/1000 [00:00<?, ?it/s]

[I 2025-12-24 19:18:54,102] Trial 4 finished with value: 0.921153033381298 and parameters: {'learning_rate': 0.08931544493767186, 'max_depth': 8, 'min_child_weight': 10, 'subsample': 0.8571073529725897, 'colsample_bytree': 0.6432433318596895, 'gamma': 3.9018012868112947e-08, 'reg_alpha': 0.009308003517763501, 'reg_lambda': 0.09977264536697893, 'n_estimators': 171}. Best is trial 4 with value: 0.921153033381298.
[I 2025-12-24 19:18:54,788] Trial 0 finished with value: 0.9208091233606659 and parameters: {'learning_rate': 0.15, 'max_depth': 9, 'min_child_weight': 7, 'subsample': 1.0, 'colsample_bytree': 0.8, 'gamma': 0.1, 'reg_alpha': 1e-08, 'reg_lambda': 1.0, 'n_estimators': 200}. Best is trial 4 with value: 0.921153033381298.
[I 2025-12-24 19:18:55,950] Trial 6 finished with value: 0.9130836409403863 and parameters: {'learning_rate': 0.046158641730617095, 'max_depth': 3, 'min_child_weight': 10, 'subsample': 0.9558696569541243, 'colsample_bytree': 0.6993352166163807, 'gamma': 0.000120792